In [12]:
from utils.functions import *
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from Preprocessing_pipelines.Outliers_pipeline import OutliersDealer
from Preprocessing_pipelines.Missing_Values_pipeline import MissingValuesDealer
from Preprocessing_pipelines.Encoding_pipeline import EncodingDealer
from Preprocessing_pipelines.Scaling_pipeline import ScalingDealer
from Preprocessing_pipelines.Feature_Selection_pipeline import FeatureSelectionDealer
from utils.Nuestra_Pipeline import *
data = pd.read_csv("../data/data_cleaned.csv")
pd.set_option("display.max_columns", None)

In [2]:
data = data.sample(frac=1).reset_index(drop=True)

In [3]:
data

,carID,Brand,model,car_age,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners
0,42633,Mercedes,Slk,10.0,11999,Semi-Auto,50027.0,Diesel,145.0,56.5,2.0,81.0,2.0
1,8738,BMW,8 series,5.0,18450,Semi-Auto,5882.0,Petrol,145.0,54.3,1.5,43.0,0.0
2,43926,Mercedes,A class,7.0,17500,Automatic,34476.0,Diesel,NaN,64.2,1.6,76.0,0.0
3,6534,Audi,Q2,7.0,20490,Semi-Auto,15992.0,Diesel,145.0,58.9,2.0,45.0,4.0
4,5719,Audi,A6,5.0,44950,Semi-Auto,8000.0,Diesel,150.0,35.8,3.0,50.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
75968,72002,VW,Golf,9.0,10290,Manual,48760.0,Petrol,30.0,53.3,1.4,93.0,4.0
75969,25200,Ford,Mondeo,5.0,20000,Manual,33.0,Diesel,145.0,65.7,2.0,47.0,1.0
75970,20657,Ford,Mustang,6.0,29000,Automatic,11475.0,Petrol,145.0,28.8,2.3,30.0,1.0
75971,4617,Audi,A5,5.0,31990,Automatic,4298.0,Diesel,145.0,45.6,2.0,49.0,4.0


In [14]:
X = data.drop("price", axis= 1)
y = data["price"]

In [4]:
X_train = data.drop("price", axis=1)
y_train = data["price"]

In [5]:
imputer = MissingValuesDealer(imputation_method="knn_modelwise",      
        strategy_cat="most_frequent",            
        knn_neighbors=5,
        random_state=42,
        knn_scaling_method= "standard")

outliers_removel = OutliersDealer()

encoder = EncodingDealer()

scaler = ScalingDealer()



In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [7]:
imputer.fit(X_train)

,imputation_method,'knn_modelwise'
,simple_strategy_num,'mean'
,strategy_cat,'most_frequent'
,fill_value,None
,knn_neighbors,5
,random_state,42
,knn_scaling_method,'standard'
,min_model_size_for_knn,15


In [9]:
imputed_X =imputer.transform(X_train)

In [10]:
imputed_X_val= imputer.transform(X_val)

In [11]:
imputed_X_val.isna().sum()

carID             0
Brand             0
model             0
car_age           0
transmission      0
mileage           0
fuelType          0
tax               0
mpg               0
engineSize        0
paintQuality%     0
previousOwners    0
dtype: int64

Testing with Nuestra pipeline

In [13]:
rf = RandomForestRegressor(n_jobs=3)
dt = DecisionTreeRegressor()
kn = KNeighborsRegressor()
et = ExtraTreesRegressor()

pipeline = NuestraPipeline(OutliersDealer(), 
                           MissingValuesDealer(),
                           EncodingDealer(), 
                           ScalingDealer(),
                           FeatureSelectionDealer(),
                           rf)

parameters = [


    
    
    # Random Forest
    {
              "imputer__imputation_method": ["knn_modelwise"],
              
}]

mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=parameters,  
    n_iter=15,                       
    scoring=mae_scorer,
    refit="mae_scorer",               
    verbose=3,
    n_jobs=3,
    random_state=42,                  
    cv=5                              
)

In [15]:
random_search.fit(X,y)

C:\Users\andre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 1 is smaller than n_iter=15. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits


,estimator,NuestraPipeli...ctionDealer())
,param_distributions,[{'imputer__imputation_method': ['knn_modelwise']}]
,n_iter,15
,scoring,make_scorer(m...hod='predict')
,n_jobs,3
,refit,'mae_scorer'
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan
